In [6]:
import os, random, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2
USE_IMAGENET_WEIGHTS = True
RESUME = False

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请在 Kaggle 的 Notebook Settings -> Accelerator 中选择 GPU。"
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# =========================
# 实验配置：X0 Baseline
# =========================
EXPERIMENT = "X0_Baseline"
USE_EDGE = False
EDGE_LAMBDA = 0.0

if EXPERIMENT == "X0_Baseline" and (USE_EDGE or EDGE_LAMBDA != 0.0):
    raise ValueError("X0 必须设置 USE_EDGE=False，EDGE_LAMBDA=0.0")

if EXPERIMENT == "X1_Edge_NoSupervision" and (
    not USE_EDGE or EDGE_LAMBDA != 0.0
):
    raise ValueError("X1 必须设置 USE_EDGE=True，EDGE_LAMBDA=0.0")

if EXPERIMENT == "X2_Edge_Supervision" and (
    not USE_EDGE or EDGE_LAMBDA <= 0.0
):
    raise ValueError("X2 必须设置 USE_EDGE=True，EDGE_LAMBDA>0.0")

OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT)
print("USE_EDGE:", USE_EDGE)
print("EDGE_LAMBDA:", EDGE_LAMBDA)
print("OUT_ROOT:", OUT_ROOT)

# =========================
# 自动寻找 WHU 数据
# =========================
MANUAL_WHU_ROOT = ""

def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        root = Path(MANUAL_WHU_ROOT)
        candidates = [(
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
        )]
    else:
        candidates = []

        for top in (Path("/kaggle/input"), Path("/kaggle/working")):
            if not top.exists():
                continue

            for train_dir in top.rglob("train"):
                candidates.append((
                    train_dir / "image",
                    train_dir / "label",
                    train_dir.parent / "val" / "image",
                    train_dir.parent / "val" / "label",
                ))

    valid = [
        c for c in candidates
        if all(p.is_dir() for p in c)
    ]

    if not valid:
        raise FileNotFoundError(
            "没有找到 WHU 数据。请把 MANUAL_WHU_ROOT 改为包含 train 和 val 的目录。"
        )

    valid.sort(
        key=lambda c: (
            "whu" not in str(c[0]).lower(),
            len(str(c[0]))
        )
    )

    return valid[0]

TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL = locate_whu()

print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)

# =========================
# Dataset
# =========================
class WHUDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, img_size=256):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff",
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file() and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(f"没有找到图像：{self.img_dir}")

        missing = [
            name for name in self.names
            if not (self.lbl_dir / name).exists()
        ]

        if missing:
            raise FileNotFoundError(
                f"缺少同名标签，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.names)

    @staticmethod
    def make_whu_edge(mask_np):
        mask_u8 = mask_np.astype(np.uint8) * 255
        kernel = np.ones((3, 3), np.uint8)
        dilated = cv2.dilate(
            mask_u8,
            kernel,
            iterations=1
        )

        return (
            (dilated - mask_u8) > 0
        ).astype(np.float32)

    def __getitem__(self, idx):
        name = self.names[idx]

        image = Image.open(
            self.img_dir / name
        ).convert("RGB")

        label = Image.open(
            self.lbl_dir / name
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        image_tensor = T.ToTensor()(image)

        label_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        edge_np = self.make_whu_edge(label_np)

        return (
            image_tensor,
            torch.from_numpy(label_np).long(),
            torch.from_numpy(edge_np).float(),
            name,
        )

train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))

# =========================
# 模型
# =========================
class EdgeBranch(nn.Module):
    def __init__(self, in_channels=3, out_channels=32):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.edge_head = nn.Conv2d(
            out_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):
        feat = self.conv(x)
        edge_logits = self.edge_head(feat)
        return feat, edge_logits


class ResNetSemanticBranch(nn.Module):
    def __init__(self, out_channels=64):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if USE_IMAGENET_WEIGHTS
            else None
        )

        try:
            resnet = models.resnet18(
                weights=weights
            )
        except Exception as exc:
            raise RuntimeError(
                "ResNet18 权重加载失败。"
                "请打开 Kaggle Internet，或者将 "
                "USE_IMAGENET_WEIGHTS=False。"
            ) from exc

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.smooth = nn.Sequential(
            nn.Conv2d(
                512,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.smooth(feat)

        return F.interpolate(
            feat,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


class EdgeEnhancedNet(nn.Module):
    def __init__(self, use_edge=True):
        super().__init__()

        self.use_edge = use_edge
        self.semantic_branch = ResNetSemanticBranch(64)

        if use_edge:
            self.edge_branch = EdgeBranch(3, 32)
            fusion_in = 96
        else:
            self.edge_branch = None
            fusion_in = 64

        self.fusion = nn.Sequential(
            nn.Conv2d(
                fusion_in,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                64,
                2,
                kernel_size=1
            ),
        )

    def forward(self, x):
        semantic_feat = self.semantic_branch(x)
        edge_logits = None

        if self.use_edge:
            edge_feat, edge_logits = self.edge_branch(x)
            fused = torch.cat(
                [semantic_feat, edge_feat],
                dim=1
            )
        else:
            fused = semantic_feat

        seg_logits = self.fusion(fused)

        return seg_logits, edge_logits


# =========================
# 损失函数
# =========================
ce_loss = nn.CrossEntropyLoss()
edge_bce = nn.BCEWithLogitsLoss()

def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(dim=(1, 2))

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2.0 * inter + eps
    ) / (
        denom + eps
    )

    return (
        1.0 - dice
    ).mean()


# =========================
# 评价指标
# =========================
def get_boundary(mask):
    mask_u8 = mask.astype(np.uint8) * 255
    kernel = np.ones((3, 3), np.uint8)

    dilated = cv2.dilate(
        mask_u8,
        kernel,
        iterations=1
    )

    eroded = cv2.erode(
        mask_u8,
        kernel,
        iterations=1
    )

    return (
        (dilated - eroded) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gt_boundary = get_boundary(gt)
    pred_boundary = get_boundary(pred)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary
    ).sum()

    union = np.logical_or(
        gt_boundary,
        pred_boundary
    ).sum()

    boundary_iou = (
        intersection / (union + 1e-8)
    )

    kernel = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gt_dilated = cv2.dilate(
        gt_boundary,
        kernel
    )

    pred_dilated = cv2.dilate(
        pred_boundary,
        kernel
    )

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1
    ).sum()

    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1
    ).sum()

    precision = matched_pred / (
        (pred_boundary == 1).sum()
        + 1e-8
    )

    recall = matched_gt / (
        (gt_boundary == 1).sum()
        + 1e-8
    )

    boundary_f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return float(boundary_f1), float(boundary_iou)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    boundary_f1_list = []
    boundary_iou_list = []

    for images, labels, _, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        seg_logits, _ = model(images)

        pred = torch.argmax(
            seg_logits,
            dim=1
        ).cpu().numpy().astype(np.uint8)

        gt = labels.numpy().astype(np.uint8)

        tp += np.logical_and(
            pred == 1,
            gt == 1
        ).sum()

        fp += np.logical_and(
            pred == 1,
            gt == 0
        ).sum()

        fn += np.logical_and(
            pred == 0,
            gt == 1
        ).sum()

        tn += np.logical_and(
            pred == 0,
            gt == 0
        ).sum()

        for p, g in zip(pred, gt):
            bf1, biou = boundary_metrics(
                g,
                p,
                tolerance=2
            )

            boundary_f1_list.append(bf1)
            boundary_iou_list.append(biou)

    building_iou = tp / (
        tp + fp + fn + 1e-8
    )

    background_iou = tn / (
        tn + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    f1 = 2 * precision * recall / (
        precision + recall + 1e-8
    )

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(
            (building_iou + background_iou) / 2
        ),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(
            np.mean(boundary_f1_list)
        ),
        "BoundaryIoU": float(
            np.mean(boundary_iou_list)
        ),
    }


def count_params_m(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) / 1e6


# =========================
# 训练
# =========================
seed_everything(SEED)

model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

last_path = CKPT_DIR / f"{EXPERIMENT}_last.pth"
best_path = CKPT_DIR / f"{EXPERIMENT}_best.pth"

best_iou = -1.0
history = []
start_epoch = 1

if RESUME and last_path.exists():
    checkpoint = torch.load(
        last_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    best_iou = float(
        checkpoint.get(
            "best_iou",
            -1.0
        )
    )

    history = checkpoint.get(
        "history",
        []
    )

    start_epoch = int(
        checkpoint["epoch"]
    ) + 1

    print(
        f"从第 {start_epoch - 1} 轮恢复，"
        f"当前最佳 IoU={best_iou:.6f}"
    )

for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_edge_loss = 0.0
    batches = 0

    t0 = time.time()

    for images, labels, edges, _ in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        edges = edges.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=True
        ):
            seg_logits, edge_logits = model(images)

            loss_seg = (
                ce_loss(seg_logits, labels)
                + dice_loss_building(
                    seg_logits,
                    labels
                )
            )

            if USE_EDGE and EDGE_LAMBDA > 0.0:
                loss_edge = edge_bce(
                    edge_logits.squeeze(1),
                    edges
                )
            else:
                loss_edge = torch.zeros(
                    (),
                    device=DEVICE
                )

            loss = (
                loss_seg
                + EDGE_LAMBDA * loss_edge
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.item()
        )

        total_seg_loss += float(
            loss_seg.item()
        )

        total_edge_loss += float(
            loss_edge.item()
        )

        batches += 1

    avg_loss = total_loss / max(
        batches,
        1
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_seg_loss": (
            total_seg_loss / max(batches, 1)
        ),
        "train_edge_loss": (
            total_edge_loss / max(batches, 1)
        ),
        **val_metrics,
    })

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]

        torch.save(
            model.state_dict(),
            best_path
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        last_path
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"IoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"F1={val_metrics['F1'] * 100:.2f}% | "
        f"BF1={val_metrics['BoundaryF1'] * 100:.2f}%"
    )

# =========================
# 最终评估
# =========================
if not best_path.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{best_path}"
    )

best_model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        best_path,
        map_location=DEVICE
    )
)

metrics = evaluate(
    best_model,
    val_loader
)

metrics_pct = {
    k: v * 100.0
    for k, v in metrics.items()
}

config = {
    "experiment": EXPERIMENT,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "use_edge": USE_EDGE,
    "edge_lambda": EDGE_LAMBDA,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": (
        "BCEWithLogitsLoss"
        if EDGE_LAMBDA > 0
        else "disabled"
    ),
    "whu_edge_label": "3x3 Dilate(M)-M",
    "best_checkpoint_rule": (
        "maximum validation Building IoU"
    ),
    "postprocessing": "none",
    "imagenet_pretrained": USE_IMAGENET_WEIGHTS,
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(RESULT_DIR / f"{EXPERIMENT}_metrics.json").write_text(
    json.dumps(
        metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

report_lines = [
    f"WHU formal ablation experiment: {EXPERIMENT}",
    "=" * 70,
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint validation metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v in metrics_pct.items()
)

report = "\n".join(report_lines)

(RESULT_DIR / f"{EXPERIMENT}_report.txt").write_text(
    report,
    encoding="utf-8"
)

# =========================
# 保存训练曲线
# =========================
hist = pd.DataFrame(history)

plt.figure(figsize=(7.2, 4.6))
plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"{EXPERIMENT} Training Curve")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

# =========================
# 保存指标图
# =========================
metric_names = [
    "BuildingIoU",
    "F1",
    "BoundaryF1",
    "BoundaryIoU",
]

metric_values = [
    metrics_pct[k]
    for k in metric_names
]

plt.figure(figsize=(7.2, 4.6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title(f"{EXPERIMENT} Validation Metrics")

for bar, value in zip(
    bars,
    metric_values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_metrics.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

# =========================
# 打包结果
# =========================
archive_path = Path(
    "/kaggle/working"
) / f"{EXPERIMENT}_artifacts.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )

print("\n" + report)
print("\n结果压缩包：", archive_path)

Device: cuda
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Experiment: X0_Baseline
USE_EDGE: False
EDGE_LAMBDA: 0.0
OUT_ROOT: /kaggle/working/WHU_X0_Baseline_final
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Train samples: 4736
Val samples  : 1036
Epoch 01/50 | Loss=0.9084 | IoU=48.21% | F1=65.06% | BF1=28.21%
Epoch 02/50 | Loss=0.7809 | IoU=52.31% | F1=68.69% | BF1=31.07%
Epoch 03/50 | Loss=0.7285 | IoU=56.48% | F1=72.19% | BF1=37.09%
Epoch 04/50 | Loss=0.6856 | IoU=54.90% | F1=70.88% | BF1=37.02%
Epoch 05/50 | Loss=0.6648 | IoU=59.33% | F1=74.47% | BF1=40.08%
Epoch 06/50 | Loss=0.6568 | IoU=58.68% | F1=73.96% | BF1=38.81%
Epoch 07/50 | Loss=0.6188 | IoU=60.11% | F1=75.09% | BF1=40.38%
Epoch 08/50 | Loss=0.6055 | IoU=59.44% 

In [7]:
import os, random, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models
import matplotlib.pyplot as plt

# =========================
# 0. 固定实验参数
# =========================
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

USE_IMAGENET_WEIGHTS = True
RESUME = False

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请在 Kaggle Notebook Settings -> Accelerator 中选择 GPU。"
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# =========================
# 1. X1 实验配置
# =========================
EXPERIMENT = "X1_Edge_NoSupervision"

# X1：保留 EdgeBranch，关闭边缘监督
USE_EDGE = True
EDGE_LAMBDA = 0.0

if not USE_EDGE:
    raise ValueError("X1 必须设置 USE_EDGE=True")

if EDGE_LAMBDA != 0.0:
    raise ValueError("X1 必须设置 EDGE_LAMBDA=0.0")

OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT)
print("USE_EDGE:", USE_EDGE)
print("EDGE_LAMBDA:", EDGE_LAMBDA)
print("OUT_ROOT:", OUT_ROOT)

# =========================
# 2. 自动寻找 WHU 数据
# =========================
MANUAL_WHU_ROOT = ""

def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        root = Path(MANUAL_WHU_ROOT)

        candidates = [(
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
        )]
    else:
        candidates = []

        for top in (
            Path("/kaggle/input"),
            Path("/kaggle/working")
        ):
            if not top.exists():
                continue

            for train_dir in top.rglob("train"):
                candidates.append((
                    train_dir / "image",
                    train_dir / "label",
                    train_dir.parent / "val" / "image",
                    train_dir.parent / "val" / "label",
                ))

    valid = [
        c for c in candidates
        if all(p.is_dir() for p in c)
    ]

    if not valid:
        raise FileNotFoundError(
            "没有找到 WHU 数据。请把 MANUAL_WHU_ROOT 改为包含 train 和 val 的目录。"
        )

    valid.sort(
        key=lambda c: (
            "whu" not in str(c[0]).lower(),
            len(str(c[0]))
        )
    )

    return valid[0]


TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL = locate_whu()

print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)

# =========================
# 3. Dataset
# =========================
class WHUDataset(Dataset):
    def __init__(
        self,
        img_dir,
        lbl_dir,
        img_size=256
    ):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff",
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(
                f"没有找到图像：{self.img_dir}"
            )

        missing = [
            name
            for name in self.names
            if not (self.lbl_dir / name).exists()
        ]

        if missing:
            raise FileNotFoundError(
                f"缺少同名标签，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.names)

    @staticmethod
    def make_whu_edge(mask_np):
        mask_u8 = mask_np.astype(np.uint8) * 255

        kernel = np.ones(
            (3, 3),
            np.uint8
        )

        dilated = cv2.dilate(
            mask_u8,
            kernel,
            iterations=1
        )

        return (
            (dilated - mask_u8) > 0
        ).astype(np.float32)

    def __getitem__(self, idx):
        name = self.names[idx]

        image = Image.open(
            self.img_dir / name
        ).convert("RGB")

        label = Image.open(
            self.lbl_dir / name
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        image_tensor = T.ToTensor()(image)

        label_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        edge_np = self.make_whu_edge(label_np)

        return (
            image_tensor,
            torch.from_numpy(label_np).long(),
            torch.from_numpy(edge_np).float(),
            name,
        )


train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))

# =========================
# 4. 模型结构
# =========================
class EdgeBranch(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=32
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.edge_head = nn.Conv2d(
            out_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):
        feat = self.conv(x)
        edge_logits = self.edge_head(feat)
        return feat, edge_logits


class ResNetSemanticBranch(nn.Module):
    def __init__(
        self,
        out_channels=64
    ):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if USE_IMAGENET_WEIGHTS
            else None
        )

        try:
            resnet = models.resnet18(
                weights=weights
            )
        except Exception as exc:
            raise RuntimeError(
                "ResNet18 权重加载失败。请打开 Kaggle Internet，"
                "或者将 USE_IMAGENET_WEIGHTS=False。"
            ) from exc

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.smooth = nn.Sequential(
            nn.Conv2d(
                512,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.smooth(feat)

        return F.interpolate(
            feat,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


class EdgeEnhancedNet(nn.Module):
    def __init__(
        self,
        use_edge=True
    ):
        super().__init__()

        self.use_edge = use_edge
        self.semantic_branch = ResNetSemanticBranch(64)

        if use_edge:
            self.edge_branch = EdgeBranch(
                in_channels=3,
                out_channels=32
            )
            fusion_in = 96
        else:
            self.edge_branch = None
            fusion_in = 64

        self.fusion = nn.Sequential(
            nn.Conv2d(
                fusion_in,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                2,
                kernel_size=1
            ),
        )

    def forward(self, x):
        semantic_feat = self.semantic_branch(x)

        edge_logits = None

        if self.use_edge:
            edge_feat, edge_logits = self.edge_branch(x)

            fused = torch.cat(
                [semantic_feat, edge_feat],
                dim=1
            )
        else:
            fused = semantic_feat

        seg_logits = self.fusion(fused)

        return seg_logits, edge_logits


# =========================
# 5. 损失函数
# =========================
ce_loss = nn.CrossEntropyLoss()
edge_bce = nn.BCEWithLogitsLoss()

def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(dim=(1, 2))

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2.0 * inter + eps
    ) / (
        denom + eps
    )

    return (
        1.0 - dice
    ).mean()


# =========================
# 6. 评价指标
# =========================
def get_boundary(mask):
    mask_u8 = mask.astype(np.uint8) * 255

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    dilated = cv2.dilate(
        mask_u8,
        kernel,
        iterations=1
    )

    eroded = cv2.erode(
        mask_u8,
        kernel,
        iterations=1
    )

    return (
        (dilated - eroded) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gt_boundary = get_boundary(gt)
    pred_boundary = get_boundary(pred)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary
    ).sum()

    union = np.logical_or(
        gt_boundary,
        pred_boundary
    ).sum()

    boundary_iou = (
        intersection / (union + 1e-8)
    )

    kernel = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gt_dilated = cv2.dilate(
        gt_boundary,
        kernel
    )

    pred_dilated = cv2.dilate(
        pred_boundary,
        kernel
    )

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1
    ).sum()

    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1
    ).sum()

    precision = matched_pred / (
        (pred_boundary == 1).sum()
        + 1e-8
    )

    recall = matched_gt / (
        (gt_boundary == 1).sum()
        + 1e-8
    )

    boundary_f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return float(boundary_f1), float(boundary_iou)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    boundary_f1_list = []
    boundary_iou_list = []

    for images, labels, _, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        seg_logits, _ = model(images)

        pred = torch.argmax(
            seg_logits,
            dim=1
        ).cpu().numpy().astype(np.uint8)

        gt = labels.numpy().astype(np.uint8)

        tp += np.logical_and(
            pred == 1,
            gt == 1
        ).sum()

        fp += np.logical_and(
            pred == 1,
            gt == 0
        ).sum()

        fn += np.logical_and(
            pred == 0,
            gt == 1
        ).sum()

        tn += np.logical_and(
            pred == 0,
            gt == 0
        ).sum()

        for p, g in zip(pred, gt):
            bf1, biou = boundary_metrics(
                g,
                p,
                tolerance=2
            )

            boundary_f1_list.append(bf1)
            boundary_iou_list.append(biou)

    building_iou = tp / (
        tp + fp + fn + 1e-8
    )

    background_iou = tn / (
        tn + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    f1 = 2 * precision * recall / (
        precision + recall + 1e-8
    )

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(
            (building_iou + background_iou) / 2
        ),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(
            np.mean(boundary_f1_list)
        ),
        "BoundaryIoU": float(
            np.mean(boundary_iou_list)
        ),
    }


def count_params_m(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) / 1e6


# =========================
# 7. 开始训练
# =========================
seed_everything(SEED)

model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

last_path = CKPT_DIR / f"{EXPERIMENT}_last.pth"
best_path = CKPT_DIR / f"{EXPERIMENT}_best.pth"

best_iou = -1.0
history = []
start_epoch = 1

if RESUME and last_path.exists():
    checkpoint = torch.load(
        last_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    best_iou = float(
        checkpoint.get(
            "best_iou",
            -1.0
        )
    )

    history = checkpoint.get(
        "history",
        []
    )

    start_epoch = int(
        checkpoint["epoch"]
    ) + 1

    print(
        f"从第 {start_epoch - 1} 轮恢复，"
        f"当前最佳 IoU={best_iou:.6f}"
    )

for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_edge_loss = 0.0
    batches = 0

    t0 = time.time()

    for images, labels, edges, _ in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        edges = edges.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=True
        ):
            seg_logits, edge_logits = model(images)

            loss_seg = (
                ce_loss(seg_logits, labels)
                + dice_loss_building(
                    seg_logits,
                    labels
                )
            )

            # X1 中保留 EdgeBranch，
            # 但不计算边缘监督损失
            loss_edge = torch.zeros(
                (),
                device=DEVICE
            )

            loss = loss_seg

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.item()
        )

        total_seg_loss += float(
            loss_seg.item()
        )

        total_edge_loss += float(
            loss_edge.item()
        )

        batches += 1

    avg_loss = total_loss / max(
        batches,
        1
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_seg_loss": (
            total_seg_loss / max(batches, 1)
        ),
        "train_edge_loss": (
            total_edge_loss / max(batches, 1)
        ),
        **val_metrics,
    })

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]

        torch.save(
            model.state_dict(),
            best_path
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        last_path
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"IoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"F1={val_metrics['F1'] * 100:.2f}% | "
        f"BF1={val_metrics['BoundaryF1'] * 100:.2f}%"
    )

# =========================
# 8. 最终评估
# =========================
if not best_path.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{best_path}"
    )

best_model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        best_path,
        map_location=DEVICE
    )
)

metrics = evaluate(
    best_model,
    val_loader
)

metrics_pct = {
    k: v * 100.0
    for k, v in metrics.items()
}

config = {
    "experiment": EXPERIMENT,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "use_edge": USE_EDGE,
    "edge_lambda": EDGE_LAMBDA,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": "disabled",
    "whu_edge_label": "3x3 Dilate(M)-M",
    "best_checkpoint_rule": (
        "maximum validation Building IoU"
    ),
    "postprocessing": "none",
    "imagenet_pretrained": USE_IMAGENET_WEIGHTS,
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(RESULT_DIR / f"{EXPERIMENT}_metrics.json").write_text(
    json.dumps(
        metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

report_lines = [
    f"WHU formal ablation experiment: {EXPERIMENT}",
    "=" * 70,
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint validation metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v in metrics_pct.items()
)

report = "\n".join(report_lines)

(RESULT_DIR / f"{EXPERIMENT}_report.txt").write_text(
    report,
    encoding="utf-8"
)

# =========================
# 9. 保存训练曲线
# =========================
hist = pd.DataFrame(history)

plt.figure(figsize=(7.2, 4.6))

plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"{EXPERIMENT} Training Curve")
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# =========================
# 10. 保存指标图
# =========================
metric_names = [
    "BuildingIoU",
    "F1",
    "BoundaryF1",
    "BoundaryIoU",
]

metric_values = [
    metrics_pct[k]
    for k in metric_names
]

plt.figure(figsize=(7.2, 4.6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title(f"{EXPERIMENT} Validation Metrics")

for bar, value in zip(
    bars,
    metric_values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# =========================
# 11. 打包结果
# =========================
archive_path = Path(
    "/kaggle/working"
) / f"{EXPERIMENT}_artifacts.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )

print("\n" + report)
print("\n结果压缩包：", archive_path)

Device: cuda
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Experiment: X1_Edge_NoSupervision
USE_EDGE: True
EDGE_LAMBDA: 0.0
OUT_ROOT: /kaggle/working/WHU_X1_Edge_NoSupervision_final
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Train samples: 4736
Val samples  : 1036
Epoch 01/50 | Loss=0.5479 | IoU=73.60% | F1=84.79% | BF1=52.35%
Epoch 02/50 | Loss=0.4294 | IoU=67.26% | F1=80.43% | BF1=50.72%
Epoch 03/50 | Loss=0.3953 | IoU=74.44% | F1=85.35% | BF1=53.98%
Epoch 04/50 | Loss=0.3718 | IoU=78.85% | F1=88.17% | BF1=56.59%
Epoch 05/50 | Loss=0.3547 | IoU=79.23% | F1=88.41% | BF1=56.73%
Epoch 06/50 | Loss=0.3427 | IoU=80.91% | F1=89.45% | BF1=57.64%
Epoch 07/50 | Loss=0.3222 | IoU=81.83% | F1=90.01% | BF1=59.46%
Epoch 08/50 | Loss=0

In [1]:
import os, random, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models

import matplotlib.pyplot as plt


# ============================================================
# 0. 固定实验参数
# ============================================================
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

USE_IMAGENET_WEIGHTS = True
RESUME = False


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请在 Kaggle Notebook Settings -> Accelerator 中选择 GPU。"
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 1. X2 实验配置
# ============================================================
EXPERIMENT = "X2_Edge_Supervision"

# X2：保留 EdgeBranch，并启用边缘监督
USE_EDGE = True
EDGE_LAMBDA = 0.2

if not USE_EDGE:
    raise ValueError("X2 必须设置 USE_EDGE=True")

if EDGE_LAMBDA <= 0:
    raise ValueError("X2 必须设置 EDGE_LAMBDA>0")


OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT)
print("USE_EDGE:", USE_EDGE)
print("EDGE_LAMBDA:", EDGE_LAMBDA)
print("OUT_ROOT:", OUT_ROOT)


# ============================================================
# 2. 自动寻找 WHU 数据
# ============================================================
MANUAL_WHU_ROOT = ""


def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        root = Path(MANUAL_WHU_ROOT)

        candidates = [(
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
        )]
    else:
        candidates = []

        for top in (
            Path("/kaggle/input"),
            Path("/kaggle/working"),
        ):
            if not top.exists():
                continue

            for train_dir in top.rglob("train"):
                candidates.append((
                    train_dir / "image",
                    train_dir / "label",
                    train_dir.parent / "val" / "image",
                    train_dir.parent / "val" / "label",
                ))

    valid = [
        c for c in candidates
        if all(p.is_dir() for p in c)
    ]

    if not valid:
        raise FileNotFoundError(
            "没有找到 WHU 数据。请把 MANUAL_WHU_ROOT 改为包含 train 和 val 的目录。"
        )

    valid.sort(
        key=lambda c: (
            "whu" not in str(c[0]).lower(),
            len(str(c[0]))
        )
    )

    return valid[0]


TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL = locate_whu()

print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)


# ============================================================
# 3. Dataset
# ============================================================
class WHUDataset(Dataset):
    def __init__(
        self,
        img_dir,
        lbl_dir,
        img_size=256
    ):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff",
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(
                f"没有找到图像：{self.img_dir}"
            )

        missing = [
            name
            for name in self.names
            if not (self.lbl_dir / name).exists()
        ]

        if missing:
            raise FileNotFoundError(
                f"缺少同名标签，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.names)

    @staticmethod
    def make_whu_edge(mask_np):
        mask_u8 = mask_np.astype(np.uint8) * 255

        kernel = np.ones(
            (3, 3),
            np.uint8
        )

        dilated = cv2.dilate(
            mask_u8,
            kernel,
            iterations=1
        )

        return (
            (dilated - mask_u8) > 0
        ).astype(np.float32)

    def __getitem__(self, idx):
        name = self.names[idx]

        image = Image.open(
            self.img_dir / name
        ).convert("RGB")

        label = Image.open(
            self.lbl_dir / name
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        image_tensor = T.ToTensor()(image)

        label_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        edge_np = self.make_whu_edge(label_np)

        return (
            image_tensor,
            torch.from_numpy(label_np).long(),
            torch.from_numpy(edge_np).float(),
            name,
        )


train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))


# ============================================================
# 4. 模型结构
# ============================================================
class EdgeBranch(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=32
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.edge_head = nn.Conv2d(
            out_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):
        feat = self.conv(x)
        edge_logits = self.edge_head(feat)
        return feat, edge_logits


class ResNetSemanticBranch(nn.Module):
    def __init__(
        self,
        out_channels=64
    ):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if USE_IMAGENET_WEIGHTS
            else None
        )

        try:
            resnet = models.resnet18(
                weights=weights
            )
        except Exception as exc:
            raise RuntimeError(
                "ResNet18 权重加载失败。请打开 Kaggle Internet，"
                "或者将 USE_IMAGENET_WEIGHTS=False。"
            ) from exc

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.smooth = nn.Sequential(
            nn.Conv2d(
                512,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.smooth(feat)

        return F.interpolate(
            feat,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


class EdgeEnhancedNet(nn.Module):
    def __init__(
        self,
        use_edge=True
    ):
        super().__init__()

        self.use_edge = use_edge
        self.semantic_branch = ResNetSemanticBranch(64)

        if use_edge:
            self.edge_branch = EdgeBranch(
                in_channels=3,
                out_channels=32
            )
            fusion_in = 96
        else:
            self.edge_branch = None
            fusion_in = 64

        self.fusion = nn.Sequential(
            nn.Conv2d(
                fusion_in,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                2,
                kernel_size=1
            ),
        )

    def forward(self, x):
        semantic_feat = self.semantic_branch(x)

        edge_logits = None

        if self.use_edge:
            edge_feat, edge_logits = self.edge_branch(x)

            fused = torch.cat(
                [semantic_feat, edge_feat],
                dim=1
            )
        else:
            fused = semantic_feat

        seg_logits = self.fusion(fused)

        return seg_logits, edge_logits


# ============================================================
# 5. 损失函数
# ============================================================
ce_loss = nn.CrossEntropyLoss()
edge_bce = nn.BCEWithLogitsLoss()


def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(dim=(1, 2))

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2.0 * inter + eps
    ) / (
        denom + eps
    )

    return (
        1.0 - dice
    ).mean()


# ============================================================
# 6. 评价指标
# ============================================================
def get_boundary(mask):
    mask_u8 = mask.astype(np.uint8) * 255

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    dilated = cv2.dilate(
        mask_u8,
        kernel,
        iterations=1
    )

    eroded = cv2.erode(
        mask_u8,
        kernel,
        iterations=1
    )

    return (
        (dilated - eroded) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gt_boundary = get_boundary(gt)
    pred_boundary = get_boundary(pred)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary
    ).sum()

    union = np.logical_or(
        gt_boundary,
        pred_boundary
    ).sum()

    boundary_iou = (
        intersection / (union + 1e-8)
    )

    kernel = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gt_dilated = cv2.dilate(
        gt_boundary,
        kernel
    )

    pred_dilated = cv2.dilate(
        pred_boundary,
        kernel
    )

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1
    ).sum()

    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1
    ).sum()

    precision = matched_pred / (
        (pred_boundary == 1).sum()
        + 1e-8
    )

    recall = matched_gt / (
        (gt_boundary == 1).sum()
        + 1e-8
    )

    boundary_f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return float(boundary_f1), float(boundary_iou)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    boundary_f1_list = []
    boundary_iou_list = []

    for images, labels, _, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        seg_logits, _ = model(images)

        pred = torch.argmax(
            seg_logits,
            dim=1
        ).cpu().numpy().astype(np.uint8)

        gt = labels.numpy().astype(np.uint8)

        tp += np.logical_and(
            pred == 1,
            gt == 1
        ).sum()

        fp += np.logical_and(
            pred == 1,
            gt == 0
        ).sum()

        fn += np.logical_and(
            pred == 0,
            gt == 1
        ).sum()

        tn += np.logical_and(
            pred == 0,
            gt == 0
        ).sum()

        for p, g in zip(pred, gt):
            bf1, biou = boundary_metrics(
                g,
                p,
                tolerance=2
            )

            boundary_f1_list.append(bf1)
            boundary_iou_list.append(biou)

    building_iou = tp / (
        tp + fp + fn + 1e-8
    )

    background_iou = tn / (
        tn + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    f1 = 2 * precision * recall / (
        precision + recall + 1e-8
    )

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(
            (building_iou + background_iou) / 2
        ),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(
            np.mean(boundary_f1_list)
        ),
        "BoundaryIoU": float(
            np.mean(boundary_iou_list)
        ),
    }


def count_params_m(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) / 1e6


# ============================================================
# 7. 开始训练
# ============================================================
seed_everything(SEED)

model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

last_path = CKPT_DIR / f"{EXPERIMENT}_last.pth"
best_path = CKPT_DIR / f"{EXPERIMENT}_best.pth"

best_iou = -1.0
history = []
start_epoch = 1

if RESUME and last_path.exists():
    checkpoint = torch.load(
        last_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    best_iou = float(
        checkpoint.get(
            "best_iou",
            -1.0
        )
    )

    history = checkpoint.get(
        "history",
        []
    )

    start_epoch = int(
        checkpoint["epoch"]
    ) + 1

    print(
        f"从第 {start_epoch - 1} 轮恢复，"
        f"当前最佳 IoU={best_iou:.6f}"
    )


for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_edge_loss = 0.0
    batches = 0

    t0 = time.time()

    for images, labels, edges, _ in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        edges = edges.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=True
        ):
            seg_logits, edge_logits = model(images)

            loss_seg = (
                ce_loss(seg_logits, labels)
                + dice_loss_building(
                    seg_logits,
                    labels
                )
            )

            # X2 的核心：边缘监督
            loss_edge = edge_bce(
                edge_logits.squeeze(1),
                edges
            )

            loss = (
                loss_seg
                + EDGE_LAMBDA * loss_edge
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.item()
        )

        total_seg_loss += float(
            loss_seg.item()
        )

        total_edge_loss += float(
            loss_edge.item()
        )

        batches += 1

    avg_loss = total_loss / max(
        batches,
        1
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_seg_loss": (
            total_seg_loss / max(batches, 1)
        ),
        "train_edge_loss": (
            total_edge_loss / max(batches, 1)
        ),
        **val_metrics,
    })

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]

        torch.save(
            model.state_dict(),
            best_path
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        last_path
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"IoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"F1={val_metrics['F1'] * 100:.2f}% | "
        f"BF1={val_metrics['BoundaryF1'] * 100:.2f}%"
    )


# ============================================================
# 8. 最终评估
# ============================================================
if not best_path.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{best_path}"
    )

best_model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        best_path,
        map_location=DEVICE
    )
)

metrics = evaluate(
    best_model,
    val_loader
)

metrics_pct = {
    k: v * 100.0
    for k, v in metrics.items()
}


config = {
    "experiment": EXPERIMENT,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "use_edge": USE_EDGE,
    "edge_lambda": EDGE_LAMBDA,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": "BCEWithLogitsLoss",
    "whu_edge_label": "3x3 Dilate(M)-M",
    "best_checkpoint_rule": (
        "maximum validation Building IoU"
    ),
    "postprocessing": "none",
    "imagenet_pretrained": USE_IMAGENET_WEIGHTS,
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(RESULT_DIR / f"{EXPERIMENT}_metrics.json").write_text(
    json.dumps(
        metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


report_lines = [
    f"WHU formal ablation experiment: {EXPERIMENT}",
    "=" * 70,
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint validation metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v in metrics_pct.items()
)

report = "\n".join(report_lines)

(RESULT_DIR / f"{EXPERIMENT}_report.txt").write_text(
    report,
    encoding="utf-8"
)


# ============================================================
# 9. 保存训练曲线
# ============================================================
hist = pd.DataFrame(history)

plt.figure(figsize=(7.2, 4.6))

plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"{EXPERIMENT} Training Curve")
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 10. 保存指标图
# ============================================================
metric_names = [
    "BuildingIoU",
    "F1",
    "BoundaryF1",
    "BoundaryIoU",
]

metric_values = [
    metrics_pct[k]
    for k in metric_names
]

plt.figure(figsize=(7.2, 4.6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title(f"{EXPERIMENT} Validation Metrics")

for bar, value in zip(
    bars,
    metric_values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 11. 打包结果
# ============================================================
archive_path = Path(
    "/kaggle/working"
) / f"{EXPERIMENT}_artifacts.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )

print("\n" + report)
print("\n结果压缩包：", archive_path)

Device: cuda
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Experiment: X2_Edge_Supervision
USE_EDGE: True
EDGE_LAMBDA: 0.2
OUT_ROOT: /kaggle/working/WHU_X2_Edge_Supervision_final
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Train samples: 4736
Val samples  : 1036
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 222MB/s]


Epoch 01/50 | Loss=0.5983 | IoU=72.68% | F1=84.18% | BF1=52.97%
Epoch 02/50 | Loss=0.4557 | IoU=70.84% | F1=82.93% | BF1=52.27%
Epoch 03/50 | Loss=0.4242 | IoU=74.80% | F1=85.58% | BF1=53.79%
Epoch 04/50 | Loss=0.4006 | IoU=79.67% | F1=88.69% | BF1=57.14%
Epoch 05/50 | Loss=0.3833 | IoU=56.43% | F1=72.14% | BF1=45.14%
Epoch 06/50 | Loss=0.3682 | IoU=80.40% | F1=89.13% | BF1=58.21%
Epoch 07/50 | Loss=0.3473 | IoU=80.75% | F1=89.35% | BF1=57.55%
Epoch 08/50 | Loss=0.3491 | IoU=72.45% | F1=84.03% | BF1=54.96%
Epoch 09/50 | Loss=0.3460 | IoU=81.56% | F1=89.84% | BF1=58.89%
Epoch 10/50 | Loss=0.3435 | IoU=74.25% | F1=85.22% | BF1=54.13%
Epoch 11/50 | Loss=0.3453 | IoU=80.67% | F1=89.30% | BF1=55.80%
Epoch 12/50 | Loss=0.3091 | IoU=81.35% | F1=89.71% | BF1=57.29%
Epoch 13/50 | Loss=0.2775 | IoU=79.10% | F1=88.33% | BF1=54.81%
Epoch 14/50 | Loss=0.3179 | IoU=79.35% | F1=88.48% | BF1=55.52%
Epoch 15/50 | Loss=0.3183 | IoU=74.11% | F1=85.13% | BF1=50.18%
Epoch 16/50 | Loss=0.3066 | IoU=81.98% |

In [ ]:
import os, random, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models

import matplotlib.pyplot as plt


# ============================================================
# 0. 固定实验参数
# ============================================================
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

USE_IMAGENET_WEIGHTS = True
RESUME = False


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请在 Kaggle Notebook Settings -> Accelerator 中选择 GPU。"
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 1. X2-Lambda010 实验配置
# ============================================================
EXPERIMENT = "X2_Lambda010"

# X2-Lambda010：保留 EdgeBranch，并启用边缘监督；lambda_edge=0.10
USE_EDGE = True
EDGE_LAMBDA = 0.1

if not USE_EDGE:
    raise ValueError("X2 必须设置 USE_EDGE=True")

if abs(EDGE_LAMBDA - 0.1) > 1e-12:
    raise ValueError("X2_Lambda010 必须设置 EDGE_LAMBDA=0.1")


OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT)
print("USE_EDGE:", USE_EDGE)
print("EDGE_LAMBDA:", EDGE_LAMBDA)
print("OUT_ROOT:", OUT_ROOT)


# ============================================================
# 2. 自动寻找 WHU 数据
# ============================================================
MANUAL_WHU_ROOT = ""


def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        root = Path(MANUAL_WHU_ROOT)

        candidates = [(
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
        )]
    else:
        candidates = []

        for top in (
            Path("/kaggle/input"),
            Path("/kaggle/working"),
        ):
            if not top.exists():
                continue

            for train_dir in top.rglob("train"):
                candidates.append((
                    train_dir / "image",
                    train_dir / "label",
                    train_dir.parent / "val" / "image",
                    train_dir.parent / "val" / "label",
                ))

    valid = [
        c for c in candidates
        if all(p.is_dir() for p in c)
    ]

    if not valid:
        raise FileNotFoundError(
            "没有找到 WHU 数据。请把 MANUAL_WHU_ROOT 改为包含 train 和 val 的目录。"
        )

    valid.sort(
        key=lambda c: (
            "whu" not in str(c[0]).lower(),
            len(str(c[0]))
        )
    )

    return valid[0]


TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL = locate_whu()

print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)


# ============================================================
# 3. Dataset
# ============================================================
class WHUDataset(Dataset):
    def __init__(
        self,
        img_dir,
        lbl_dir,
        img_size=256
    ):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff",
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(
                f"没有找到图像：{self.img_dir}"
            )

        missing = [
            name
            for name in self.names
            if not (self.lbl_dir / name).exists()
        ]

        if missing:
            raise FileNotFoundError(
                f"缺少同名标签，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.names)

    @staticmethod
    def make_whu_edge(mask_np):
        mask_u8 = mask_np.astype(np.uint8) * 255

        kernel = np.ones(
            (3, 3),
            np.uint8
        )

        dilated = cv2.dilate(
            mask_u8,
            kernel,
            iterations=1
        )

        return (
            (dilated - mask_u8) > 0
        ).astype(np.float32)

    def __getitem__(self, idx):
        name = self.names[idx]

        image = Image.open(
            self.img_dir / name
        ).convert("RGB")

        label = Image.open(
            self.lbl_dir / name
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        image_tensor = T.ToTensor()(image)

        label_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        edge_np = self.make_whu_edge(label_np)

        return (
            image_tensor,
            torch.from_numpy(label_np).long(),
            torch.from_numpy(edge_np).float(),
            name,
        )


train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))


# ============================================================
# 4. 模型结构
# ============================================================
class EdgeBranch(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=32
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.edge_head = nn.Conv2d(
            out_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):
        feat = self.conv(x)
        edge_logits = self.edge_head(feat)
        return feat, edge_logits


class ResNetSemanticBranch(nn.Module):
    def __init__(
        self,
        out_channels=64
    ):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if USE_IMAGENET_WEIGHTS
            else None
        )

        try:
            resnet = models.resnet18(
                weights=weights
            )
        except Exception as exc:
            raise RuntimeError(
                "ResNet18 权重加载失败。请打开 Kaggle Internet，"
                "或者将 USE_IMAGENET_WEIGHTS=False。"
            ) from exc

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.smooth = nn.Sequential(
            nn.Conv2d(
                512,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.smooth(feat)

        return F.interpolate(
            feat,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


class EdgeEnhancedNet(nn.Module):
    def __init__(
        self,
        use_edge=True
    ):
        super().__init__()

        self.use_edge = use_edge
        self.semantic_branch = ResNetSemanticBranch(64)

        if use_edge:
            self.edge_branch = EdgeBranch(
                in_channels=3,
                out_channels=32
            )
            fusion_in = 96
        else:
            self.edge_branch = None
            fusion_in = 64

        self.fusion = nn.Sequential(
            nn.Conv2d(
                fusion_in,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                2,
                kernel_size=1
            ),
        )

    def forward(self, x):
        semantic_feat = self.semantic_branch(x)

        edge_logits = None

        if self.use_edge:
            edge_feat, edge_logits = self.edge_branch(x)

            fused = torch.cat(
                [semantic_feat, edge_feat],
                dim=1
            )
        else:
            fused = semantic_feat

        seg_logits = self.fusion(fused)

        return seg_logits, edge_logits


# ============================================================
# 5. 损失函数
# ============================================================
ce_loss = nn.CrossEntropyLoss()
edge_bce = nn.BCEWithLogitsLoss()


def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(dim=(1, 2))

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2.0 * inter + eps
    ) / (
        denom + eps
    )

    return (
        1.0 - dice
    ).mean()


# ============================================================
# 6. 评价指标
# ============================================================
def get_boundary(mask):
    mask_u8 = mask.astype(np.uint8) * 255

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    dilated = cv2.dilate(
        mask_u8,
        kernel,
        iterations=1
    )

    eroded = cv2.erode(
        mask_u8,
        kernel,
        iterations=1
    )

    return (
        (dilated - eroded) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gt_boundary = get_boundary(gt)
    pred_boundary = get_boundary(pred)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary
    ).sum()

    union = np.logical_or(
        gt_boundary,
        pred_boundary
    ).sum()

    boundary_iou = (
        intersection / (union + 1e-8)
    )

    kernel = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gt_dilated = cv2.dilate(
        gt_boundary,
        kernel
    )

    pred_dilated = cv2.dilate(
        pred_boundary,
        kernel
    )

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1
    ).sum()

    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1
    ).sum()

    precision = matched_pred / (
        (pred_boundary == 1).sum()
        + 1e-8
    )

    recall = matched_gt / (
        (gt_boundary == 1).sum()
        + 1e-8
    )

    boundary_f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return float(boundary_f1), float(boundary_iou)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    boundary_f1_list = []
    boundary_iou_list = []

    for images, labels, _, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        seg_logits, _ = model(images)

        pred = torch.argmax(
            seg_logits,
            dim=1
        ).cpu().numpy().astype(np.uint8)

        gt = labels.numpy().astype(np.uint8)

        tp += np.logical_and(
            pred == 1,
            gt == 1
        ).sum()

        fp += np.logical_and(
            pred == 1,
            gt == 0
        ).sum()

        fn += np.logical_and(
            pred == 0,
            gt == 1
        ).sum()

        tn += np.logical_and(
            pred == 0,
            gt == 0
        ).sum()

        for p, g in zip(pred, gt):
            bf1, biou = boundary_metrics(
                g,
                p,
                tolerance=2
            )

            boundary_f1_list.append(bf1)
            boundary_iou_list.append(biou)

    building_iou = tp / (
        tp + fp + fn + 1e-8
    )

    background_iou = tn / (
        tn + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    f1 = 2 * precision * recall / (
        precision + recall + 1e-8
    )

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(
            (building_iou + background_iou) / 2
        ),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(
            np.mean(boundary_f1_list)
        ),
        "BoundaryIoU": float(
            np.mean(boundary_iou_list)
        ),
    }


def count_params_m(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) / 1e6


# ============================================================
# 7. 开始训练
# ============================================================
seed_everything(SEED)

model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

last_path = CKPT_DIR / f"{EXPERIMENT}_last.pth"
best_path = CKPT_DIR / f"{EXPERIMENT}_best.pth"

best_iou = -1.0
history = []
start_epoch = 1

if RESUME and last_path.exists():
    checkpoint = torch.load(
        last_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    best_iou = float(
        checkpoint.get(
            "best_iou",
            -1.0
        )
    )

    history = checkpoint.get(
        "history",
        []
    )

    start_epoch = int(
        checkpoint["epoch"]
    ) + 1

    print(
        f"从第 {start_epoch - 1} 轮恢复，"
        f"当前最佳 IoU={best_iou:.6f}"
    )


for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_edge_loss = 0.0
    batches = 0

    t0 = time.time()

    for images, labels, edges, _ in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        edges = edges.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=True
        ):
            seg_logits, edge_logits = model(images)

            loss_seg = (
                ce_loss(seg_logits, labels)
                + dice_loss_building(
                    seg_logits,
                    labels
                )
            )

            # X2-Lambda010 的核心：边缘监督，lambda_edge=0.10
            loss_edge = edge_bce(
                edge_logits.squeeze(1),
                edges
            )

            loss = (
                loss_seg
                + EDGE_LAMBDA * loss_edge
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.item()
        )

        total_seg_loss += float(
            loss_seg.item()
        )

        total_edge_loss += float(
            loss_edge.item()
        )

        batches += 1

    avg_loss = total_loss / max(
        batches,
        1
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_seg_loss": (
            total_seg_loss / max(batches, 1)
        ),
        "train_edge_loss": (
            total_edge_loss / max(batches, 1)
        ),
        **val_metrics,
    })

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]

        torch.save(
            model.state_dict(),
            best_path
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        last_path
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"IoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"F1={val_metrics['F1'] * 100:.2f}% | "
        f"BF1={val_metrics['BoundaryF1'] * 100:.2f}%"
    )


# ============================================================
# 8. 最终评估
# ============================================================
if not best_path.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{best_path}"
    )

best_model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        best_path,
        map_location=DEVICE
    )
)

metrics = evaluate(
    best_model,
    val_loader
)

metrics_pct = {
    k: v * 100.0
    for k, v in metrics.items()
}


config = {
    "experiment": EXPERIMENT,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "use_edge": USE_EDGE,
    "edge_lambda": EDGE_LAMBDA,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": "BCEWithLogitsLoss",
    "whu_edge_label": "3x3 Dilate(M)-M",
    "best_checkpoint_rule": (
        "maximum validation Building IoU"
    ),
    "postprocessing": "none",
    "imagenet_pretrained": USE_IMAGENET_WEIGHTS,
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(RESULT_DIR / f"{EXPERIMENT}_metrics.json").write_text(
    json.dumps(
        metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


report_lines = [
    f"WHU formal ablation experiment: {EXPERIMENT}",
    "=" * 70,
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint validation metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v in metrics_pct.items()
)

report = "\n".join(report_lines)

(RESULT_DIR / f"{EXPERIMENT}_report.txt").write_text(
    report,
    encoding="utf-8"
)


# ============================================================
# 9. 保存训练曲线
# ============================================================
hist = pd.DataFrame(history)

plt.figure(figsize=(7.2, 4.6))

plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"{EXPERIMENT} Training Curve")
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 10. 保存指标图
# ============================================================
metric_names = [
    "BuildingIoU",
    "F1",
    "BoundaryF1",
    "BoundaryIoU",
]

metric_values = [
    metrics_pct[k]
    for k in metric_names
]

plt.figure(figsize=(7.2, 4.6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title(f"{EXPERIMENT} Validation Metrics")

for bar, value in zip(
    bars,
    metric_values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 11. 打包结果
# ============================================================
archive_path = Path(
    "/kaggle/working"
) / f"{EXPERIMENT}_artifacts.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )

print("\n" + report)
print("\n结果压缩包：", archive_path)

Device: cuda
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Experiment: X2_Lambda010
USE_EDGE: True
EDGE_LAMBDA: 0.1
OUT_ROOT: /kaggle/working/WHU_X2_Lambda010_final
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Train samples: 4736
Val samples  : 1036
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s] 
